In [1]:

# ================================================================
# PROJECT 14: WHAT-IF BUSINESS SIMULATOR AI
# Interactive Strategic Scenario Planning Studio
# ================================================================

!pip -q install gradio pandas numpy plotly

import gradio as gr
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

# -------------------- DESIGN SYSTEM --------------------

BG = "#F8F7F2"
WHITE = "#FFFFFF"
INK = "#34384B"
MUTED = "#7B8093"
LAVENDER = "#E8E1FA"
LILAC = "#9279C9"
MINT = "#DDF2E5"
GREEN = "#49886C"
PEACH = "#FBE4D6"
CORAL = "#D78269"
BLUE = "#DFECFA"
BLUE_DARK = "#668DBD"
YELLOW = "#F8EFCF"
BORDER = "#E9E6E1"

CSS = """
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=Manrope:wght@400;500;600;700;800&display=swap');

.gradio-container {
    max-width: 1480px !important;
    margin: auto !important;
    background: #F8F7F2 !important;
    font-family: 'DM Sans', sans-serif !important;
}
body { background: #F8F7F2 !important; }
h1, h2, h3, p { color: #34384B; }
.hero {
    background: linear-gradient(120deg, #E8E1FA 0%, #DDF2E5 55%, #DFECFA 100%);
    padding: 34px;
    border-radius: 22px;
    margin-bottom: 15px;
}
.hero h1 {
    font-family: 'Manrope', sans-serif !important;
    font-size: 38px !important;
    line-height: 1.2;
    color: #34384B !important;
    margin: 12px 0;
}
.hero p { max-width: 760px; line-height: 1.7; color: #62677B; }
.eyebrow {
    font-size: 11px;
    font-weight: 700;
    letter-spacing: 2px;
    text-transform: uppercase;
    color: #77718F;
}
.metric {
    padding: 18px;
    background: white;
    border: 1px solid #E9E6E1;
    border-radius: 16px;
    min-height: 112px;
}
.metric-label { font-size: 11px; color: #7B8093; text-transform: uppercase; letter-spacing: 1px; }
.metric-value { font-size: 28px; font-weight: 800; color: #34384B; margin-top: 8px; }
.metric-note { font-size: 12px; color: #7B8093; margin-top: 5px; }
.panel {
    padding: 17px 20px;
    border: 1px solid #E9E6E1;
    background: white;
    border-radius: 16px;
}
.insight {
    padding: 17px;
    background: white;
    border: 1px solid #E9E6E1;
    border-left: 5px solid #9279C9;
    border-radius: 12px;
    margin: 9px 0;
}
.insight h3 { margin: 0 0 7px; font-size: 16px; }
.insight p { margin: 5px 0; color: #62677B; line-height: 1.6; }
.section-heading {
    font-family: 'Manrope', sans-serif;
    font-weight: 800;
    font-size: 24px;
    margin: 14px 0 5px;
}
.note {
    background: #FFF9E9;
    border: 1px solid #F0E4BE;
    padding: 14px 16px;
    border-radius: 12px;
    color: #70664D;
    line-height: 1.6;
    font-size: 13px;
}
button.primary {
    background: #9279C9 !important;
    border-color: #9279C9 !important;
}
footer { display: none !important; }
"""

# -------------------- BASELINE COMPANY --------------------

BASE = {
    "monthly_sales": 1000,       # orders per month
    "average_price": 5000,       # currency units per order
    "employees": 50,
    "employee_capacity": 25,     # orders per employee per month
    "inventory": 1200,           # orders worth of stock
    "production_capacity": 1250, # orders per month
    "logistics_capacity": 1200,  # orders per month
    "unit_cost": 3200,
    "fixed_cost": 650000,
    "hiring_cost": 30000,        # one-time cost per new employee
    "inventory_cost": 1500,      # cost per additional stock unit
    "production_expansion_cost": 100000,
    "logistics_expansion_cost": 60000
}

# -------------------- SCENARIO ENGINE --------------------

def simulate(
    sales_growth,
    new_employees,
    extra_inventory,
    production_expansion,
    logistics_expansion,
    productivity_improvement,
    efficiency_improvement,
    hiring_strategy,
    inventory_strategy,
    production_strategy,
    logistics_strategy
):
    b = BASE.copy()

    # Use the direct controls to set the amount of each intervention.
    extra_staff = int(new_employees)
    stock_added = int(extra_inventory)
    production_added = int(production_expansion)
    logistics_added = int(logistics_expansion)

    baseline_demand = b["monthly_sales"]
    target_demand = int(round(baseline_demand * (1 + sales_growth / 100)))

    # Effective employee capacity.
    staff_capacity = (
        (b["employees"] + extra_staff) *
        b["employee_capacity"] *
        (1 + productivity_improvement / 100)
    )

    # Other capacity constraints.
    stock_capacity = b["inventory"] + stock_added
    production_capacity = b["production_capacity"] + production_added
    logistics_capacity = b["logistics_capacity"] + logistics_added

    # Deliverable demand is limited by the smallest available capacity.
    deliverable_orders = max(0, min(
        target_demand,
        int(staff_capacity),
        stock_capacity,
        production_capacity,
        logistics_capacity
    ))

    unmet_demand = max(0, target_demand - deliverable_orders)
    fulfilled_pct = (
        100 * deliverable_orders / target_demand if target_demand else 100
    )

    # Revenue and variable cost.
    revenue = deliverable_orders * b["average_price"]
    variable_cost = deliverable_orders * b["unit_cost"] * (
        1 - efficiency_improvement / 100
    )

    # Operating costs include monthly fixed costs and intervention costs.
    hiring_expense = extra_staff * b["hiring_cost"] / 12
    stock_expense = stock_added * b["inventory_cost"] / 12

    # Expansion costs are treated as monthly planning allowances here.
    production_expense = (
        b["production_expansion_cost"] * production_added /
        max(1, b["production_capacity"]) / 12
    )
    logistics_expense = (
        b["logistics_expansion_cost"] * logistics_added /
        max(1, b["logistics_capacity"]) / 12
    )

    operating_cost = (
        variable_cost + b["fixed_cost"] + hiring_expense +
        stock_expense + production_expense + logistics_expense
    )
    profit = revenue - operating_cost
    margin = 100 * profit / revenue if revenue else 0

    # Baseline financial outcome.
    base_revenue = baseline_demand * b["average_price"]
    base_cost = baseline_demand * b["unit_cost"] + b["fixed_cost"]
    base_profit = base_revenue - base_cost

    profit_change = profit - base_profit
    profit_change_pct = (
        100 * profit_change / abs(base_profit) if base_profit else 0
    )

    capacity = {
        "Employees": staff_capacity,
        "Inventory": stock_capacity,
        "Production": production_capacity,
        "Logistics": logistics_capacity
    }

    gaps = {
        "Employees": max(0, target_demand - staff_capacity),
        "Inventory": max(0, target_demand - stock_capacity),
        "Production": max(0, target_demand - production_capacity),
        "Logistics": max(0, target_demand - logistics_capacity)
    }

    bottleneck = min(capacity, key=capacity.get)

    return {
        "target_demand": target_demand,
        "delivered": deliverable_orders,
        "unmet": unmet_demand,
        "fulfilled_pct": fulfilled_pct,
        "revenue": revenue,
        "cost": operating_cost,
        "profit": profit,
        "margin": margin,
        "base_profit": base_profit,
        "profit_change": profit_change,
        "profit_change_pct": profit_change_pct,
        "capacity": capacity,
        "gaps": gaps,
        "bottleneck": bottleneck,
        "extra_staff": extra_staff,
        "stock_added": stock_added,
        "production_added": production_added,
        "logistics_added": logistics_added,
        "sales_growth": sales_growth
    }

# -------------------- CHART HELPERS --------------------

def style(fig, height=360):
    fig.update_layout(
        height=height,
        paper_bgcolor="white",
        plot_bgcolor="white",
        font=dict(family="DM Sans", color=INK),
        margin=dict(l=25, r=20, t=50, b=30),
        legend=dict(orientation="h", y=1.12, x=0),
        hoverlabel=dict(bgcolor=INK, font_color="white")
    )
    fig.update_xaxes(showgrid=False, linecolor=BORDER)
    fig.update_yaxes(gridcolor="#F0EEF0", zerolinecolor=BORDER)
    return fig

def metric(label, value, note):
    return f"""
    <div class="metric">
        <div class="metric-label">{label}</div>
        <div class="metric-value">{value}</div>
        <div class="metric-note">{note}</div>
    </div>
    """

def capacity_figure(r):
    names = list(r["capacity"].keys())
    values = [r["capacity"][n] for n in names]
    colors = [GREEN if v >= r["target_demand"] else CORAL for v in values]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=names, y=values, marker_color=colors,
        text=[f"{v:,.0f} orders" for v in values],
        textposition="outside",
        name="Available capacity"
    ))
    fig.add_hline(
        y=r["target_demand"], line_dash="dash",
        line_color=LILAC, line_width=3,
        annotation_text=f"Demand target: {r['target_demand']:,}",
        annotation_position="top left"
    )
    fig.update_layout(yaxis_title="Orders per month")
    return style(fig, 370)

def financial_figure(r):
    labels = ["Revenue", "Operating costs", "Profit"]
    values = [r["revenue"], r["cost"], r["profit"]]
    colors = [GREEN, CORAL, LILAC if r["profit"] >= 0 else CORAL]

    fig = go.Figure(go.Bar(
        x=labels, y=values, marker_color=colors,
        text=[f"{v:,.0f}" for v in values],
        textposition="outside"
    ))
    fig.update_layout(yaxis_title="Currency units / month")
    return style(fig, 350)

def scenario_comparison(r):
    scenarios = []

    # Scenario 1: no intervention.
    scenarios.append(("Current capacity", simulate(
        r["sales_growth"], 0, 0, 0, 0, 0, 0, "", "", "", ""
    )))

    # Scenario 2: add people.
    scenarios.append(("Hire 10 employees", simulate(
        r["sales_growth"], 10, 0, 0, 0, 0, 0, "", "", "", ""
    )))

    # Scenario 3: expand stock.
    scenarios.append(("Add 300 inventory", simulate(
        r["sales_growth"], 0, 300, 0, 0, 0, 0, "", "", "", ""
    )))

    # Scenario 4: expand production and logistics.
    scenarios.append(("Expand production + logistics", simulate(
        r["sales_growth"], 0, 300, 300, 300, 0, 0, "", "", "", ""
    )))

    # Scenario 5: coordinated intervention.
    scenarios.append(("Coordinated plan", simulate(
        r["sales_growth"], 10, 300, 300, 300, 10, 5, "", "", "", ""
    )))

    fig = go.Figure()
    for name, result in scenarios:
        fig.add_trace(go.Bar(
            name=name,
            x=["Profit", "Orders fulfilled"],
            y=[result["profit"], result["delivered"]],
            text=[f"{result['profit']:,.0f}", f"{result['delivered']:,.0f}"],
            textposition="auto"
        ))

    fig.update_layout(
        barmode="group",
        title="Compare alternative business decisions",
        yaxis_title="Values (profit in currency units; orders in orders)"
    )
    return style(fig, 420)

# -------------------- RECOMMENDATIONS --------------------

def recommendations(r):
    items = []
    target = r["target_demand"]

    if r["fulfilled_pct"] >= 99.9:
        items.append((
            "Demand coverage",
            "Current simulated interventions cover the sales target.",
            "Continue monitoring fulfillment, costs, and working capital."
        ))
    else:
        items.append((
            "Demand coverage gap",
            f"Approximately {r['unmet']:,} orders per month cannot be fulfilled.",
            "Address the binding capacity constraint before committing to the target."
        ))

    if r["gaps"]["Employees"] > 0:
        need = int(np.ceil(r["gaps"]["Employees"] / (
            BASE["employee_capacity"] * (1 + 0.01)
        )))
        items.append((
            "Workforce capacity",
            f"Employee capacity is short by about {r['gaps']['Employees']:.0f} orders/month.",
            "Review staffing, shifts, cross-training, and productivity before hiring."
        ))

    if r["gaps"]["Inventory"] > 0:
        items.append((
            "Inventory readiness",
            f"Inventory is short by approximately {r['gaps']['Inventory']:.0f} orders.",
            "Check supplier lead times, reorder points, and working-capital availability."
        ))

    if r["gaps"]["Production"] > 0:
        items.append((
            "Production readiness",
            f"Production capacity is short by approximately {r['gaps']['Production']:.0f} orders/month.",
            "Review shift schedules, equipment utilization, and feasible capacity expansion."
        ))

    if r["gaps"]["Logistics"] > 0:
        items.append((
            "Logistics readiness",
            f"Logistics capacity is short by approximately {r['gaps']['Logistics']:.0f} orders/month.",
            "Assess carrier capacity, warehouse throughput, and delivery commitments."
        ))

    if r["profit"] < r["base_profit"]:
        items.append((
            "Profitability warning",
            "The simulated profit is below the baseline despite the scenario changes.",
            "Revisit intervention costs, unit economics, and the margin of incremental sales."
        ))
    else:
        items.append((
            "Financial check",
            "The modeled profit is at or above the baseline.",
            "Validate assumptions, cash flow, and one-time implementation expenses."
        ))

    html = ""
    for title, evidence, action in items:
        html += f"""
        <div class="insight">
          <h3>{title}</h3>
          <p>{evidence}</p>
          <p><b>Suggested action:</b> {action}</p>
        </div>
        """

    return html

# -------------------- MAIN SIMULATION CALLBACK --------------------

def run_simulation(
    sales_growth, new_employees, extra_inventory,
    production_expansion, logistics_expansion,
    productivity_improvement, efficiency_improvement
):
    r = simulate(
        sales_growth, new_employees, extra_inventory,
        production_expansion, logistics_expansion,
        productivity_improvement, efficiency_improvement,
        "", "", "", ""
    )

    status = "TARGET SUPPORTED" if r["fulfilled_pct"] >= 99.9 else "CAPACITY GAP DETECTED"
    status_color = GREEN if r["fulfilled_pct"] >= 99.9 else CORAL

    header = f"""
    <div class="hero">
      <div class="eyebrow">SCENARIO LAB / SIMULATION COMPLETE</div>
      <h1>Your next growth target,<br>tested before launch.</h1>
      <p>The model evaluated demand, staffing, stock, production, logistics, and estimated
      operating profit using the assumptions selected in the control panel.</p>
      <div style="display:inline-block;background:white;border-radius:10px;padding:10px 14px;
                  color:{status_color};font-weight:800;">
        {status} · {r['fulfilled_pct']:.1f}% of target demand covered
      </div>
    </div>
    """

    cards = [
        metric("Demand target", f"{r['target_demand']:,}", f"+{sales_growth:.0f}% planned sales growth"),
        metric("Orders fulfilled", f"{r['delivered']:,}", f"{r['unmet']:,} orders unmet"),
        metric("Projected monthly revenue", f"{r['revenue']:,.0f}", "Currency units"),
        metric("Projected monthly profit", f"{r['profit']:,.0f}", f"Margin: {r['margin']:.1f}%")
    ]

    finance_note = f"""
    <div class="panel">
      <b>Scenario readout</b><br>
      Baseline monthly profit: {r['base_profit']:,.0f}<br>
      Simulated profit change: {r['profit_change']:+,.0f}
      ({r['profit_change_pct']:+.1f}% versus baseline)<br>
      Binding capacity: <b>{r['bottleneck']}</b><br>
      Estimated demand gap: <b>{r['unmet']:,} orders/month</b>
    </div>
    """

    return (
        header, *cards,
        capacity_figure(r),
        financial_figure(r),
        scenario_comparison(r),
        finance_note,
        recommendations(r)
    )

# -------------------- EXPORT --------------------

def export_scenario(
    sales_growth, new_employees, extra_inventory,
    production_expansion, logistics_expansion,
    productivity_improvement, efficiency_improvement
):
    r = simulate(
        sales_growth, new_employees, extra_inventory,
        production_expansion, logistics_expansion,
        productivity_improvement, efficiency_improvement,
        "", "", "", ""
    )

    rows = [
        ("Sales growth target (%)", sales_growth),
        ("Target demand (orders/month)", r["target_demand"]),
        ("Orders fulfilled", r["delivered"]),
        ("Unmet demand", r["unmet"]),
        ("Demand covered (%)", round(r["fulfilled_pct"], 2)),
        ("Monthly revenue", round(r["revenue"], 2)),
        ("Monthly operating costs", round(r["cost"], 2)),
        ("Monthly operating profit", round(r["profit"], 2)),
        ("Profit margin (%)", round(r["margin"], 2)),
        ("Baseline monthly profit", round(r["base_profit"], 2)),
        ("Profit change", round(r["profit_change"], 2)),
        ("Binding capacity", r["bottleneck"]),
        ("Added employees", new_employees),
        ("Additional inventory", extra_inventory),
        ("Additional production capacity", production_expansion),
        ("Additional logistics capacity", logistics_expansion),
        ("Productivity improvement (%)", productivity_improvement),
        ("Unit cost efficiency improvement (%)", efficiency_improvement)
    ]

    path = "/content/what_if_business_scenario.csv"
    pd.DataFrame(rows, columns=["Metric", "Value"]).to_csv(path, index=False)
    return path

# -------------------- BUILD WEBSITE --------------------

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="purple",
        secondary_hue="green",
        neutral_hue="slate"
    ),
    css=CSS,
    title="What-If Business Simulator AI"
) as app:

    gr.HTML("""
    <div style="display:flex;justify-content:space-between;align-items:center;
                padding:14px 5px;border-bottom:1px solid #E9E6E1;margin-bottom:20px;">
      <div style="font-size:13px;font-weight:800;letter-spacing:1.6px;">
        <span style="color:#9279C9;">✳</span> WHAT-IF STUDIO
      </div>
      <div style="font-size:10px;color:#7B8093;letter-spacing:1.2px;">
        STRATEGY PLANNING / BUSINESS SIMULATION
      </div>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1, min_width=280):
            gr.Markdown("## Scenario controls")
            gr.Markdown("Change assumptions and test the consequences.")

            sales_growth = gr.Slider(
                0, 50, value=20, step=5,
                label="Target sales growth (%)"
            )
            new_employees = gr.Slider(
                0, 40, value=10, step=1,
                label="Additional employees"
            )
            extra_inventory = gr.Slider(
                0, 1000, value=300, step=50,
                label="Additional inventory capacity (orders)"
            )
            production_expansion = gr.Slider(
                0, 1000, value=300, step=50,
                label="Additional production capacity (orders/month)"
            )
            logistics_expansion = gr.Slider(
                0, 1000, value=300, step=50,
                label="Additional logistics capacity (orders/month)"
            )
            productivity_improvement = gr.Slider(
                0, 30, value=10, step=5,
                label="Employee productivity improvement (%)"
            )
            efficiency_improvement = gr.Slider(
                0, 20, value=5, step=1,
                label="Unit-cost efficiency improvement (%)"
            )

            run_btn = gr.Button("Run business simulation →", variant="primary", size="lg")

            gr.Markdown("""
            <div class="note">
            <b>Default business assumptions</b><br>
            1,000 monthly orders · 50 employees · 1,200 inventory units ·
            1,250 production capacity · 1,200 logistics capacity.<br><br>
            All values are synthetic and can be modified in the code.
            </div>
            """)

        with gr.Column(scale=3, min_width=500):
            result_header = gr.HTML()

    with gr.Row():
        card1 = gr.HTML()
        card2 = gr.HTML()
        card3 = gr.HTML()
        card4 = gr.HTML()

    gr.Markdown("---")
    gr.Markdown("## 01 / Can operations support the demand?")
    gr.Markdown("The lowest available capacity determines the maximum number of orders the business can fulfill.")
    capacity_plot = gr.Plot()

    gr.Markdown("---")
    gr.Markdown("## 02 / Financial consequences")
    finance_plot = gr.Plot()
    finance_summary = gr.HTML()

    gr.Markdown("---")
    gr.Markdown("## 03 / Compare strategic decisions")
    gr.Markdown("Compare no intervention, hiring, inventory investment, capacity expansion, and a coordinated plan.")
    compare_plot = gr.Plot()

    gr.Markdown("---")
    gr.Markdown("## 04 / Recommended management actions")
    recommendation_html = gr.HTML()

    gr.Markdown("---")
    gr.Markdown("## 05 / Export your scenario")
    export_btn = gr.Button("Export scenario results as CSV")
    export_file = gr.File(label="Download scenario report")

    gr.Markdown("""
    <div class="note">
    <b>Model limitations:</b> This is a deterministic scenario simulator using synthetic
    baseline assumptions. It is not a trained forecasting model. Demand fulfillment is
    constrained by the minimum of employee capacity, inventory, production, and logistics.
    Costs are simplified estimates. Validate actual demand, inventory units, lead times,
    cash flow, and one-time investment costs before using results for real decisions.
    </div>
    """)

    sim_outputs = [
        result_header,
        card1, card2, card3, card4,
        capacity_plot, finance_plot, compare_plot,
        finance_summary, recommendation_html
    ]

    def callback(*args):
        result = run_simulation(*args)
        # The function returns header, four cards, capacity, finance,
        # comparison, finance summary, and recommendations.
        header, c1, c2, c3, c4, cap, fin, comp, summary, rec = result
        return [header, c1, c2, c3, c4, cap, fin, comp, summary, rec]

    controls = [
        sales_growth, new_employees, extra_inventory,
        production_expansion, logistics_expansion,
        productivity_improvement, efficiency_improvement
    ]

    run_btn.click(fn=callback, inputs=controls, outputs=sim_outputs)
    app.load(fn=callback, inputs=controls, outputs=sim_outputs)

    export_btn.click(
        fn=export_scenario,
        inputs=controls,
        outputs=export_file
    )

app.launch(share=True)


/tmp/ipykernel_1311/2876297382.py:533: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://71a072d27b908ee116.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
